## File Paths in Notebooks

Notebooks in subfolders need `../` to go back one level before finding files.

`"data/rides.csv"` → looks in `notebooks/data/` ❌  
`"../data/rides.csv"` → looks in `Data_Engineering/data/` ✅

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/rides.csv")
print(df.head(5))
print(df.shape)

# Finding insghts from the data

In [ ]:
total_rides = len(df)   #number of rows 
completed_rides = len(df[df["ride_status"] == "Completed"]) #number of rows where ride_status is Completed
cancelled_rides = len(df[df["ride_status"]== "Cancelled"]) #number of rows where ride_status is Cancelled
total_revenue = df[df["ride_status"]== "Completed" ]["fare"].sum() #sum of fare column where ride_status is Completed
avg_fare = df["fare"].mean() # average overall fare 
platform_profit = total_revenue * 0.1 # 10% of total revenue as platform profit

print(f"Completed rides : {completed_rides}")
print(f"Cancelled rides : {cancelled_rides}")
print(f"Total revenue : ₹{total_revenue}")
print(f"Average fare : ₹{avg_fare}")
print(f"Platform profit : ₹{platform_profit}")  

In [ ]:
print("=== Mumbai Ride Analytics ===" , f"Total rides: {total_rides}", f"Completed rides: {completed_rides}", f"Cancelled rides: {cancelled_rides}", f"Total revenue: ₹{total_revenue}", f"Average fare: ₹{avg_fare}", f"Platform profit: ₹{platform_profit}", sep="\n")

In [ ]:
print(df.dtypes)

# Checking for missing data

In [ ]:
print(df.isnull().sum().sum())

In [ ]:
df["fare"] = df["fare"].astype('int64')

# Creating a messy dataframe

In [ ]:
import numpy as np 

df_messy = df.copy()

# Introduce some missing values
df_messy.loc[2 , "fare"] = None
df_messy.loc[5 , "fare"] = None
df_messy.loc[2 , "ride_status"] = None
df_messy.loc[5 , "ride_status"] = None
df_messy.loc[3 , "fare"] = None

# Introduce some duplicate rows
df_messy = pd.concat([df_messy , df_messy.iloc[[0,4,8]]], ignore_index = True)

# Incosistent ride_status values
df_messy.loc[1 , "ride_status"] = "completed"
df_messy.loc[6 , "ride_status"] = "CANCELLED"   

# Outlier in fare
df_messy.loc[4 , "fare"] = 10000
df_messy.loc[7, "fare"] = -50

print(df_messy)


In [ ]:
print(df_messy.columns.tolist())

# Fixing Incosistent ride_status values

In [ ]:
df_messy["ride_status"] = df_messy["ride_status"].str.title().str.strip()

In [ ]:
print(df_messy["ride_status"])

# Finding Missing Values

In [ ]:
print(df_messy.isnull().sum())

In [ ]:
df_messy = df_messy.dropna(subset=["ride_status"])
print(df_messy.isnull().sum())

# Replacing missing fare's with mean

In [ ]:
df_messy["fare"] = df_messy["fare"].fillna(df_messy["fare"].mean())
print(df_messy.isnull().sum(), df_messy["fare"],  sep="\n\n")

In [ ]:
df_messy = df_messy.drop(df_messy[df_messy["fare"] < 0].index)
print(df_messy)
#print(df_messy.drop(df_messy[df_messy["fare"] > 1000].index))

# Fixing Outliners
IQR measures: how wide the normal middle data is.

Formula:  Upper Limit = Q3+1.5×IQR    &    Lower Limit = Q1−1.5×IQR

Small IQR:

tightly packed data

Large IQR:

spread-out data

In [ ]:
#print(df_messy.loc[df_messy["fare"] <= 10000].replace(10000, 1000))    #when we want to replace outliers with a specific value we already know

In [ ]:
df_messy.sort_values(by="fare" , ascending=True, inplace=True , ignore_index=True) 
print(df_messy)

In [ ]:
q1 = df_messy["fare"].quantile(0.25)
q3 = df_messy["fare"].quantile(0.75)
print(f"Q1 : {q1}", f"Q3 : {q3}", sep="\n")
IQR = q3 - q1

lower_limit = q1 - 1.5 * IQR
upper_limit = q3 + 1.5 * IQR

print(f"Lower limit : {lower_limit}" , f"Upper Limit : {upper_limit}", sep= "\n")

In [ ]:
fare_mean = df_messy["fare"].mean()   # mean is affected by outliers
fare_median = df_messy["fare"].median()     # median is not affected by outliers and gives a better representation of the central tendency of the data when outliers are present
print(f"Mean Fare : {fare_mean}" , f"Median Fare : {fare_median}", sep="\n")

In [ ]:
df_messy.loc[df_messy["fare"] > upper_limit , "fare"] = fare_median
df_messy.loc[df_messy["fare"] < lower_limit , "fare"] = fare_median
print(df_messy)